# Combinatorial Alpha Pipeline (GPT-Based)

여러 데이터셋의 0-fail seed 알파들을 **Cartesian Product**로 결합하여
**GPT를 활용한 조합 알파**를 생성하고 Brain API로 시뮬레이션하는 파이프라인입니다.

## 주요 특징 (2026-02 업데이트)

**NEW!** GPT 기반 조합 생성:
- **PowerPool Alpha 회피**: GPT에게 weight concentration을 피하도록 명시적 지시
- **operator_list.json 강제**: GPT가 허용된 operator만 사용하도록 제한
- **Concentration Warning 해결**: seed에 집중도 경고가 있던 경우 해결 전략 제시
- **Sanity Check 통합**: 생성 후 type validation으로 invalid expression 필터링

## 실행 순서

1. **Setup** (Cell 1-3): Imports, Configuration, Constants
2. **Functions** (Cell 4-9): 각 기능별 함수 정의
3. **Dataset Check** (Cell 10): 파일 존재 확인
4. **Dry Run** (Cell 11): 조합 개수 계산
5. **Brain Session** (Cell 12): API 로그인 (시뮬레이션 시에만 필요)
6. **Pipeline Execution** (Cell 13-17): 단계별 실행
   - **Step 1**: Seed Combinations (Cartesian Product)
   - **Step 2**: GPT-based Variant Generation (PowerPool 회피)
   - **Step 3**: Sanity Check (Type Validation)
   - **Step 4**: Save Results
7. **Simulation** (Cell 18): Brain API 시뮬레이션 (선택)

## GPT 프롬프트 구조

Step 2에서 사용하는 GPT 프롬프트는 다음을 포함:

- **MISSION**: PowerPool alpha 회피 목적 명시
- **SEED EXPRESSIONS**: 조합할 seed 알파들
- **OPERATOR CONSTRAINTS**: operators_list.json만 사용 강제
- **POWERPOOL AVOIDANCE STRATEGIES**: 구체적 회피 전략 (rank, zscore, group_zscore, winsorize 등)
- **CONCENTRATION WARNING**: seed에 경고가 있었다면 해결 방안 요구
- **OUTPUT FORMAT**: JSON 강제 (expression, operators_used, rationale 포함)

## 1. Imports

In [272]:
import json
import os
import re
import sys
from datetime import datetime
from pathlib import Path
from typing import Dict, List, Tuple, Optional
from itertools import product
import random
import pandas as pd

sys.path.insert(0, str(Path('.').resolve()))

import ace_lib as ace
import llm_functions as llm
from parser import tree_node

print("✓ Imports complete")

✓ Imports complete


## 2. Configuration

**중요**: 아래 설정을 실행 전에 확인하세요!

### 데이터셋 관리 방식

**AUTO 모드** (권장):
```python
AUTO_DISCOVER_DATASETS = True
DATASET_PATTERNS = ['model*.txt', 'nws*.txt']  # 찾을 패턴
DATASETS_TO_COMBINE = None  # 모든 발견된 데이터셋 사용
```

**MANUAL 모드**:
```python
AUTO_DISCOVER_DATASETS = False
MANUAL_DATASET_FILES = {...}  # 수동 지정
DATASETS_TO_COMBINE = ['model25', 'model30']  # 특정 데이터셋만
```

In [291]:
# ========== 경로 설정 ==========
SCRIPT_DIR = Path('.').resolve()
OUTPUT_DIR = SCRIPT_DIR / "results" / "combinatorial"

# ========== 데이터셋 자동 감지 설정 ==========
AUTO_DISCOVER_DATASETS = True  # True: 자동 감지, False: 수동 지정

# 자동 감지 설정 (AUTO_DISCOVER_DATASETS = True일 때)
DATASET_SEARCH_DIR = SCRIPT_DIR  # 데이터셋 파일을 찾을 디렉토리
DATASET_PATTERNS = ['model*.txt', 'nws*.txt']  # 찾을 파일 패턴들 (model25.txt, model30.txt 등)

# 수동 지정 (AUTO_DISCOVER_DATASETS = False일 때)
MANUAL_DATASET_FILES = {
    'model25': SCRIPT_DIR / 'model25.txt',
    'model30': SCRIPT_DIR / 'model30.txt',
    'model138': SCRIPT_DIR / 'model138.txt'
}

# ========== 조합할 데이터셋 선택 ==========
# 옵션 1: 특정 데이터셋만 선택 (리스트로 지정)
DATASETS_TO_COMBINE = ['model25', 'model30', 'model138']  # 또는 None (자동 감지된 모든 데이터셋)

# 옵션 2: 모든 발견된 데이터셋 사용
# DATASETS_TO_COMBINE = None  # 주석 해제 시 자동 감지된 모든 데이터셋 사용

# Resource 파일
OPERATORS_FILE = SCRIPT_DIR / 'operators_list.json'
REGION = "EUR"
UNIVERSE = "TOP2500"
DELAY = 1
DATAFIELDS_FILE = SCRIPT_DIR / f'datafield/{DELAY}/{REGION}/{UNIVERSE}/{DELAY}_{REGION}_{UNIVERSE}_total.json'

# ========== 성능 제어 옵션 ==========
MAX_COMBINATIONS = None       # None = 전수 조합, 숫자 = 샘플링 (예: 10000)
MAX_VARIANTS_PER_COMBO = 5    # 각 seed 조합당 생성할 variant 개수
RANDOM_SEED = 42              # 재현성을 위한 시드
CONCURRENCY = 3               # Brain API 동시 시뮬레이션 수
CHECK_SANITY = True           # Sanity check 활성화 여부

# ========== GPT 설정 ==========
GPT_MODEL = 'gpt-4o-mini'     # GPT 모델: 'gpt-4o-mini' (빠름/저렴) or 'gpt-4o' (고품질/비쌈)

# ========== Brain 시뮬레이션 설정 ==========
RUN_SIMULATION = True       # True로 설정 시 Brain API 시뮬레이션 실행

# ========================================================================
# 데이터셋 자동 감지 및 로드
# ========================================================================

def discover_datasets(search_dir: Path, patterns: List[str]) -> Dict[str, Path]:
    """
    지정된 디렉토리에서 패턴에 맞는 데이터셋 파일 자동 감지
    
    Returns:
        Dict[dataset_name, file_path]
    """
    discovered = {}
    
    for pattern in patterns:
        for file_path in search_dir.glob(pattern):
            if file_path.is_file():
                # 파일명에서 .txt 제거하여 dataset 이름 생성
                dataset_name = file_path.stem
                discovered[dataset_name] = file_path
    
    return discovered


# 데이터셋 파일 설정
if AUTO_DISCOVER_DATASETS:
    print(f"[AUTO-DISCOVER] Searching for datasets in: {DATASET_SEARCH_DIR}")
    print(f"[AUTO-DISCOVER] Patterns: {DATASET_PATTERNS}")
    DATASET_FILES = discover_datasets(DATASET_SEARCH_DIR, DATASET_PATTERNS)
    
    if not DATASET_FILES:
        print(f"\n⚠️  No datasets found! Check DATASET_SEARCH_DIR and DATASET_PATTERNS")
    else:
        print(f"\n✓ Discovered {len(DATASET_FILES)} dataset(s):")
        for name, path in sorted(DATASET_FILES.items()):
            exists = "✓ OK" if path.exists() else "✗ MISSING"
            print(f"  {name:15s} [{exists:10s}] {path.name}")
else:
    print(f"[MANUAL MODE] Using manually specified datasets")
    DATASET_FILES = MANUAL_DATASET_FILES
    print(f"\n✓ Loaded {len(DATASET_FILES)} dataset(s):")
    for name, path in sorted(DATASET_FILES.items()):
        exists = "✓ OK" if path.exists() else "✗ MISSING"
        print(f"  {name:15s} [{exists:10s}] {path}")

# DATASETS_TO_COMBINE 처리
if DATASETS_TO_COMBINE is None:
    # None이면 발견된 모든 데이터셋 사용
    DATASETS_TO_COMBINE = list(DATASET_FILES.keys())
    print(f"\n[AUTO] Using all discovered datasets: {DATASETS_TO_COMBINE}")
else:
    # 지정된 데이터셋만 사용 (존재 여부 검증)
    valid_datasets = []
    invalid_datasets = []
    
    for ds in DATASETS_TO_COMBINE:
        if ds in DATASET_FILES:
            valid_datasets.append(ds)
        else:
            invalid_datasets.append(ds)
    
    if invalid_datasets:
        print(f"\n⚠️  WARNING: The following datasets were not found: {invalid_datasets}")
        print(f"    Available datasets: {list(DATASET_FILES.keys())}")
    
    DATASETS_TO_COMBINE = valid_datasets
    print(f"\n[MANUAL] Selected datasets: {DATASETS_TO_COMBINE}")

# 최종 출력
print("\n" + "="*70)
print("CONFIGURATION SUMMARY")
print("="*70)
print(f"Working directory: {SCRIPT_DIR}")
print(f"Output directory:  {OUTPUT_DIR}")
print(f"")
print(f"Dataset discovery: {'AUTO' if AUTO_DISCOVER_DATASETS else 'MANUAL'}")
print(f"Total datasets available: {len(DATASET_FILES)}")
print(f"Datasets to combine: {len(DATASETS_TO_COMBINE)} → {DATASETS_TO_COMBINE}")
print(f"")
print(f"Max combinations: {MAX_COMBINATIONS if MAX_COMBINATIONS else 'UNLIMITED (Full Cartesian Product)'}")
print(f"Variants per combination: {MAX_VARIANTS_PER_COMBO}")
print(f"GPT Model: {GPT_MODEL}")
print(f"Sanity check: {'ON' if CHECK_SANITY else 'OFF'}")
print(f"Simulation: {'ON' if RUN_SIMULATION else 'OFF'}")
print("="*70)

[AUTO-DISCOVER] Searching for datasets in: C:\Users\adg01\llm_alpha_gen\llm_alpha_gen
[AUTO-DISCOVER] Patterns: ['model*.txt', 'nws*.txt']

✓ Discovered 2 dataset(s):
  model25         [✓ OK      ] model25.txt
  model30         [✓ OK      ] model30.txt

⚠️  WARNING: The following datasets were not found: ['model138']
    Available datasets: ['model25', 'model30']

[MANUAL] Selected datasets: ['model25', 'model30']

CONFIGURATION SUMMARY
Working directory: C:\Users\adg01\llm_alpha_gen\llm_alpha_gen
Output directory:  C:\Users\adg01\llm_alpha_gen\llm_alpha_gen\results\combinatorial

Dataset discovery: AUTO
Total datasets available: 2
Datasets to combine: 2 → ['model25', 'model30']

Max combinations: UNLIMITED (Full Cartesian Product)
Variants per combination: 5
GPT Model: gpt-4o-mini
Sanity check: ON
Simulation: ON


## 3. Constants & Patterns

In [274]:
# Regex patterns
ZERO_FAIL_BLOCK_PATTERN = re.compile(
    r"--- #(\d+) \| FAIL: 0[^-]*---\n"
    r"ID: ([A-Za-z0-9]+)\n"
    r"Region: ([A-Z]+), Universe: ([A-Z0-9]+)\n"
    r"Sharpe: ([\d.-]+), Fitness: ([\d.-]+), Turnover: ([\d.-]+)\n"
    r"Expression: (.+?)\n",
    re.DOTALL
)

OPERATOR_PATTERN = re.compile(r'([a-z_]+)\s*\(', re.IGNORECASE)
JSON_CODE_BLOCK_PATTERN = re.compile(r'```(?:json)?\s*(.*?)\s*```', re.DOTALL)

# Operator categories for variant generation
COMBINATION_OPERATORS = {
    'arithmetic': ['add', 'subtract', 'multiply', 'divide', 'min', 'max'],
    'normalization': ['rank', 'zscore', 'quantile', 'normalize', 'scale'],
    'group': ['group_zscore', 'group_rank', 'group_neutralize', 'group_scale'],
    'tail_handling': ['winsorize', 'tail', 'pasteurize'],
}

print("✓ Constants defined")

✓ Constants defined


## 4. Sanity Check Functions

Type validation using parser.py tree structure

In [275]:
def return_type(node, operators, datafields):
    """Determine the return type of a tree node"""
    if node.node_type == 'operator':
        return operators[node.value]['output']

    if node.node_type == 'datafield':
        return datafields[node.value]['type']

    if node.node_type in ['number', 'string']:
        return "NUMBER"

    if node.node_type == 'special_argument':
        return "SPECIAL_ARGUMENT"


def check_input(operator_inputs, children_types, _debug=False):
    """Check if operator inputs match children types"""
    if _debug:
        print(operator_inputs)

    while len(operator_inputs) != 0:
        if operator_inputs[0] == []:
            operator_inputs.pop(0)
            children_types.pop(0)
        elif children_types[0] in operator_inputs[0]:
            operator_inputs.pop(0)
            children_types.pop(0)
        else:
            return False

    return True


def sanity_checker(exp: str, operators: dict, datafields: dict, _debug=False) -> Tuple[bool, Optional[str]]:
    """
    Validate expression using tree parsing and type checking

    Returns:
        (is_valid, error_message)
    """
    try:
        exp_tree = tree_node(exp)

        for node in [node for node in exp_tree.collect_all_nodes() if node.node_type == "operator"]:
            operator_name = node.value

            if operator_name not in operators:
                return False, f"Unknown operator: {operator_name}"

            operator_input_types = eval(operators[operator_name]['input'])
            children_return_types = [return_type(x, operators, datafields) for x in node.children]

            if not check_input(operator_input_types, children_return_types):
                if _debug:
                    print(f"Type mismatch for {operator_name}: expected {operator_input_types}, got {children_return_types}")
                return False, f"Type mismatch in operator: {operator_name}"

        final_type = return_type(exp_tree, operators, datafields)
        if final_type != "MATRIX":
            return False, f"Final output type is {final_type}, not MATRIX"

        return True, None

    except Exception as e:
        return False, f"Parsing error: {str(e)}"

print("✓ Sanity check functions defined")

✓ Sanity check functions defined


## 5. Dataset Parsing Functions

In [276]:
def parse_zero_fail_alphas_from_file(filepath: Path) -> List[Dict]:
    """
    Parse dataset file and extract zero-fail alphas

    Returns:
        List of dicts with keys: block_num, id, sharpe, fitness, turnover, expression
    """
    if not filepath.exists():
        print(f"[WARNING] File not found: {filepath}")
        return []

    content = filepath.read_text(encoding="utf-8")
    results = []

    for match in ZERO_FAIL_BLOCK_PATTERN.finditer(content):
        results.append({
            "block_num": int(match.group(1)),
            "id": match.group(2),
            "sharpe": float(match.group(5)),
            "fitness": float(match.group(6)),
            "turnover": float(match.group(7)),
            "expression": match.group(8).strip(),
        })

    return results


def load_zero_fail_alphas(dataset_files: Dict[str, Path]) -> Dict[str, List[Dict]]:
    """
    Load zero-fail alphas from dataset files

    Returns:
        Dict mapping dataset name to list of zero-fail alphas
    """
    all_alphas = {}

    for dataset_name, filepath in dataset_files.items():
        alphas = parse_zero_fail_alphas_from_file(filepath)
        all_alphas[dataset_name] = alphas
        print(f"[INFO] {dataset_name}: {len(alphas)} zero-fail alphas loaded")

    return all_alphas

print("✓ Dataset parsing functions defined")

✓ Dataset parsing functions defined


## 6. Cartesian Product Combination Functions

In [277]:
def generate_all_combinations(
    all_alphas: Dict[str, List[Dict]],
    dataset_names: List[str],
    max_combinations: Optional[int] = None,
    random_seed: Optional[int] = None
) -> List[Dict]:
    """
    Generate Cartesian product of zero-fail seeds from specified datasets

    Returns:
        List of combination dicts with structure:
        {
            'datasets': ['mdl25', 'mdl30', ...],
            'seeds': [alpha1_dict, alpha2_dict, ...],
            'expressions': [expr1, expr2, ...],
            'metadata': {...}
        }
    """
    if random_seed is not None:
        random.seed(random_seed)

    # Extract seed lists for each dataset
    seed_lists = []
    valid_dataset_names = []

    for dataset_name in dataset_names:
        if dataset_name not in all_alphas:
            print(f"[WARNING] Dataset {dataset_name} not found, skipping")
            continue

        seeds = all_alphas[dataset_name]
        if not seeds:
            print(f"[WARNING] Dataset {dataset_name} has no seeds, skipping")
            continue

        seed_lists.append(seeds)
        valid_dataset_names.append(dataset_name)

    if len(seed_lists) < 2:
        print(f"[ERROR] Need at least 2 datasets with seeds, got {len(seed_lists)}")
        return []

    # Calculate total combinations
    total_combinations = 1
    for seeds in seed_lists:
        total_combinations *= len(seeds)

    print(f"[INFO] Generating Cartesian product from {len(valid_dataset_names)} datasets:")
    for i, dataset_name in enumerate(valid_dataset_names):
        print(f"  - {dataset_name}: {len(seed_lists[i])} seeds")
    print(f"[INFO] Total combinations: {total_combinations:,}")

    # Generate combinations
    all_combinations = []

    if max_combinations is not None and total_combinations > max_combinations:
        print(f"[INFO] Sampling {max_combinations:,} combinations from {total_combinations:,}")

        # Random sampling
        sampled_count = 0
        for combination_tuple in product(*seed_lists):
            if random.random() < (max_combinations / total_combinations):
                all_combinations.append({
                    'datasets': valid_dataset_names,
                    'seeds': list(combination_tuple),
                    'expressions': [seed['expression'] for seed in combination_tuple],
                    'metadata': {
                        'seed_ids': [seed.get('id', 'N/A') for seed in combination_tuple],
                        'sharpes': [seed.get('sharpe', 0) for seed in combination_tuple],
                        'fitnesses': [seed.get('fitness', 0) for seed in combination_tuple],
                    }
                })
                sampled_count += 1
                if sampled_count >= max_combinations:
                    break
    else:
        # Full Cartesian product
        for combination_tuple in product(*seed_lists):
            all_combinations.append({
                'datasets': valid_dataset_names,
                'seeds': list(combination_tuple),
                'expressions': [seed['expression'] for seed in combination_tuple],
                'metadata': {
                    'seed_ids': [seed.get('id', 'N/A') for seed in combination_tuple],
                    'sharpes': [seed.get('sharpe', 0) for seed in combination_tuple],
                    'fitnesses': [seed.get('fitness', 0) for seed in combination_tuple],
                }
            })

    print(f"[INFO] Generated {len(all_combinations):,} seed combinations")
    return all_combinations

print("✓ Cartesian product functions defined")

✓ Cartesian product functions defined


## 7. Variant Generation Functions

In [278]:
# ============================================================
# GPT-Based Variant Generation Functions
# ============================================================

def format_operators_compact(operators_dict: dict) -> str:
    """
    Format operators_list.json into a compact string for LLM prompt.
    Groups operators by category.
    """
    ops_by_category = {}
    for op_name, op_info in operators_dict.items():
        cat = op_info.get('category', 'Other')
        if cat not in ops_by_category:
            ops_by_category[cat] = []
        ops_by_category[cat].append(op_info.get('definition', op_name))
    
    lines = []
    for cat, defs in sorted(ops_by_category.items()):
        lines.append(f"[{cat}] " + " | ".join(defs[:20]))  # Limit to first 20 per category
    return "\n".join(lines)


def build_combination_prompt(
    seed_combination: Dict,
    operators_dict: dict,
    max_variants: int = 5,
    model: str = 'gpt-4o-mini'
) -> Tuple[str, str]:
    """
    Build GPT prompt for generating combination alphas from seed expressions.
    
    Returns:
        (prompt, json_schema)
    """
    seed_exprs = seed_combination['expressions']
    datasets_used = seed_combination['datasets']
    metadata = seed_combination.get('metadata', {})
    
    # Check if any seed had concentration warnings
    # (This would come from parsing .txt files - for now, we'll add placeholder logic)
    # In production, you'd parse the Status line from .txt to detect warnings
    has_concentration_warning = False  # TODO: implement warning detection from seed metadata
    
    # Format operators for prompt
    operators_compact = format_operators_compact(operators_dict)
    
    # Format seed expressions with dataset labels
    seed_list_str = "\n".join([
        f"  {i+1}. [{datasets_used[i]}] {expr}"
        for i, expr in enumerate(seed_exprs)
    ])
    
    json_schema = '''{
  "combinations": [
    {
      "expression": "...", 
      "operators_used": ["...","..."],
      "datasets_used": ["model25","model30","model138"],
      "idea": "...",
      "rationale": {
        "why_this_helps_powerpool": "...",
        "how_it_reduces_concentration": "...",
        "notes_on_seed_warnings": "..."
      }
    }
  ]
}'''
    
    concentration_warning_section = ""
    if has_concentration_warning:
        concentration_warning_section = """
<CONCENTRATION_WARNING>
⚠️  Some seed alphas in this combination previously had warnings:
- "Weight is too strongly concentrated"
- "Too few instruments are assigned weight"

Your combined alpha MUST address these issues by:
1. Using operators that spread weight more evenly (zscore, rank, quantile, etc.)
2. Avoiding hard thresholds or conditions that create sparse weight distributions
3. Using group normalization (group_zscore with bucket grouping) to distribute across peer groups
4. Using tail operators (winsorize, pasteurize) to limit extreme weights
5. Combining multiple signals to broaden instrument coverage

IMPORTANT: The final combined alpha should assign weight to MORE instruments than the individual seeds.
</CONCENTRATION_WARNING>
"""
    
    prompt = f"""
<MISSION>
You are tasked with combining {len(seed_exprs)} seed alpha expressions into {max_variants} NEW combined alpha expressions.

**PRIMARY GOAL**: Avoid PowerPool alpha classification.
PowerPool alphas are rejected because they concentrate weight too heavily in a small number of instruments.

Your combined alphas must:
1. Distribute weight broadly across many instruments
2. Avoid concentration in a few stocks
3. Use operators from ALLOWED_OPERATORS only
4. Follow Brain FASTEXPR syntax
</MISSION>

<SEED_EXPRESSIONS>
These are the seed alpha expressions to combine (DO NOT modify them, only wrap/combine):
{seed_list_str}

Datasets: {', '.join(datasets_used)}
</SEED_EXPRESSIONS>

<OPERATOR_CONSTRAINTS>
You MUST ONLY use operators from the list below.
NEVER invent new operators or use operators not in this list.

If an operator is not listed below, it DOES NOT EXIST in the Brain platform.

ALLOWED_OPERATORS:
{operators_compact}

⚠️  CRITICAL: Unknown operators will cause simulation failures!
⚠️  Double-check every operator against the ALLOWED_OPERATORS list above.
</OPERATOR_CONSTRAINTS>

<POWERPOOL_AVOIDANCE_STRATEGIES>
To avoid PowerPool classification, use these strategies in your combinations:

1. **Cross-Sectional Normalization** (RECOMMENDED):
   - rank(combined_expression) - ranks signals across universe
   - zscore(combined_expression) - standardizes to zero mean
   - quantile(combined_expression) - maps to quantiles
   - These spread weight more evenly across instruments

2. **Group-Based Normalization** (HIGHLY EFFECTIVE):
   - group_zscore(expr, bucket(rank(expr), buckets=N)) where N=5..20
   - Normalizes within peer groups, reducing concentration
   - ⚠️  NOTE: 'industry', 'sector', 'subindustry' DO NOT EXIST in Brain
   - MUST use bucket(rank(...), buckets=N) for grouping

3. **Tail Handling** (REDUCES EXTREME WEIGHTS):
   - winsorize(expr) - clips extreme values
   - pasteurize(expr) - limits outliers
   - tail(expr, lower, upper) - trims tails

4. **Weighted Combinations** (BALANCE MULTIPLE SIGNALS):
   - add(rank(seed1), rank(seed2), ...) - equal weighted ranked signals
   - zscore(add(zscore(seed1), zscore(seed2))) - normalized sum
   - Combining multiple signals broadens coverage

5. **AVOID**:
   - Hard thresholds: if_else, condition operators
   - Sparse operations that create zero weights for many instruments
   - Single-signal dominance: ensure all seeds contribute
</POWERPOOL_AVOIDANCE_STRATEGIES>
{concentration_warning_section}
<STRUCTURAL_RULES>
RULE1: DO NOT modify seed expressions themselves (treat as black boxes)
RULE2: You may only WRAP or COMBINE seed expressions with operators
RULE3: Maximum expression depth: reasonable (no excessive nesting)
RULE4: All operators MUST come from ALLOWED_OPERATORS list
RULE5: Final output type must be MATRIX (most operators preserve this)
RULE6: Seed expressions can be referenced by their full text as shown above
RULE7: DO NOT use 'cap', 'industry', 'sector', 'subindustry' - they don't exist
RULE8: For grouping, use bucket(rank(...), buckets=N) with N between 5 and 20
</STRUCTURAL_RULES>

<DIVERSITY_REQUIREMENTS>
Generate {max_variants} DIFFERENT combination strategies:
- Vary the operators used
- Try different normalization approaches
- Mix arithmetic combinations with statistical normalization
- Some should be simple (2-3 operators), some more complex
- Each should have a distinct rationale for why it helps avoid PowerPool
</DIVERSITY_REQUIREMENTS>

<OUTPUT_FORMAT>
You MUST respond with ONLY a valid JSON object (no other text).
Do not include markdown code blocks or explanations.

Required JSON structure:
{json_schema}

Fields explanation:
- expression: The combined FASTEXPR formula using seed expressions
- operators_used: List of operator names from ALLOWED_OPERATORS (verify each one!)
- datasets_used: {datasets_used}
- idea: Brief description of the combination strategy
- rationale.why_this_helps_powerpool: Explain how this avoids concentrated weights
- rationale.how_it_reduces_concentration: Specific mechanisms (e.g., "rank() spreads weights evenly")
- rationale.notes_on_seed_warnings: If concentration warnings exist, explain how you addressed them
</OUTPUT_FORMAT>

<EXAMPLES>
Good combination patterns:
1. rank(add(SEED1, SEED2, SEED3))
   → Combines signals then ranks, spreading weight evenly

2. zscore(add(zscore(SEED1), zscore(SEED2)))
   → Normalizes each seed then combines, avoiding dominance

3. group_zscore(add(SEED1, SEED2), bucket(rank(add(SEED1, SEED2)), buckets=10))
   → Group normalization within deciles

4. winsorize(zscore(add(SEED1, SEED2, SEED3)))
   → Combines, normalizes, then clips extremes

Bad patterns (DO NOT USE):
× divide(SEED1, cap) - 'cap' doesn't exist
× group_rank(SEED1, industry) - 'industry' doesn't exist  
× my_custom_operator(SEED1) - not in ALLOWED_OPERATORS
× if_else(SEED1 > 0, SEED1, 0) - creates sparse weights
</EXAMPLES>

Generate exactly {max_variants} diverse combination strategies now.
</MISSION>
""".strip()
    
    return prompt, json_schema


def parse_gpt_combination_response(response: str) -> List[Dict]:
    """
    Parse GPT response into list of combination dicts.
    Uses robust JSON extraction (first '{' to last '}').
    
    Returns:
        List of combination dicts, or empty list if parsing fails
    """
    if not response:
        return []
    
    # Extract JSON using robust parser
    json_str = llm.cut_first_to_last_brace(response)
    if not json_str:
        print("[ERROR] Could not find JSON braces in GPT response")
        return []
    
    try:
        data = json.loads(json_str)
        combinations = data.get('combinations', [])
        
        # Validate structure
        valid_combinations = []
        for combo in combinations:
            if 'expression' in combo and 'operators_used' in combo:
                valid_combinations.append(combo)
            else:
                print(f"[WARN] Skipping invalid combination (missing required fields): {combo}")
        
        return valid_combinations
    
    except json.JSONDecodeError as e:
        print(f"[ERROR] JSON parsing failed: {e}")
        print(f"[ERROR] Attempted to parse: {json_str[:200]}...")
        return []


def generate_combination_variants(
    seed_combination: Dict,
    operators: dict,
    max_variants_per_combo: int = 5,
    random_seed: Optional[int] = None,
    model: str = 'gpt-4o-mini'
) -> List[Dict]:
    """
    Generate expression variants from a seed combination using GPT.
    
    **NEW**: This function now uses GPT to generate combinations instead of rule-based patterns.
    
    Returns:
        List of variant dicts with structure:
        {
            'expression': 'combined_expression',
            'variant_type': 'gpt_generated',
            'idea': 'description from GPT',
            'combination_info': original seed_combination,
            'operators_used': [list of operator names],
            'rationale': {dict with PowerPool avoidance rationale}
        }
    """
    # Build prompt
    prompt, json_schema = build_combination_prompt(
        seed_combination,
        operators,
        max_variants=max_variants_per_combo,
        model=model
    )
    
    # Call GPT via llm_functions
    try:
        print(f"    [GPT] Generating {max_variants_per_combo} combinations...")
        gpt_response = llm.call_llm_stream(prompt, json_schema, model=model)
    except Exception as e:
        print(f"    [ERROR] GPT call failed: {e}")
        return []
    
    # Parse response
    combinations = parse_gpt_combination_response(gpt_response)
    
    if not combinations:
        print(f"    [WARN] GPT returned 0 valid combinations")
        return []
    
    # Convert to variant format (compatible with existing pipeline)
    variants = []
    for combo in combinations:
        variants.append({
            'expression': combo.get('expression', ''),
            'variant_type': 'gpt_generated',
            'idea': combo.get('idea', 'N/A'),
            'combination_info': seed_combination,
            'operators_used': combo.get('operators_used', []),
            'rationale': combo.get('rationale', {}),
            'datasets_used': combo.get('datasets_used', seed_combination['datasets'])
        })
    
    print(f"    [GPT] ✓ Generated {len(variants)} combinations")
    return variants


print("✓ GPT-based variant generation functions defined")

✓ GPT-based variant generation functions defined


## 8. Output Functions

In [279]:
def save_variants_to_txt(variants: List[Dict], filepath: Path, dataset_names: List[str]):
    """Save variants to human-readable text file (supports GPT-generated variants)"""
    with open(filepath, 'w', encoding='utf-8') as f:
        f.write("=" * 80 + "\n")
        f.write(f"COMBINATORIAL ALPHAS: {' x '.join(dataset_names)}\n")
        f.write(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write(f"Method: GPT-based combination (PowerPool avoidance)\n")
        f.write(f"Total variants: {len(variants)}\n")
        f.write("=" * 80 + "\n\n")

        for i, variant in enumerate(variants, 1):
            combo_info = variant.get('combination_info', {})
            metadata = combo_info.get('metadata', {})
            rationale = variant.get('rationale', {})

            f.write(f"--- #{i} | {variant.get('variant_type', 'N/A')} ---\n")
            f.write(f"Idea: {variant.get('idea', 'N/A')}\n")
            f.write(f"Expression: {variant['expression']}\n")
            f.write(f"Operators Used: {', '.join(variant.get('operators_used', []))}\n")
            f.write(f"Source Datasets: {', '.join(combo_info.get('datasets', []))}\n")
            f.write(f"Seed IDs: {', '.join(metadata.get('seed_ids', []))}\n")
            f.write(f"Source Sharpes: {', '.join(map(str, metadata.get('sharpes', [])))}\n")
            f.write(f"Source Fitnesses: {', '.join(map(str, metadata.get('fitnesses', [])))}\n")
            
            # PowerPool avoidance rationale (if GPT-generated)
            if rationale:
                f.write(f"\nRationale:\n")
                f.write(f"  - PowerPool Avoidance: {rationale.get('why_this_helps_powerpool', 'N/A')}\n")
                f.write(f"  - Concentration Reduction: {rationale.get('how_it_reduces_concentration', 'N/A')}\n")
                if rationale.get('notes_on_seed_warnings'):
                    f.write(f"  - Seed Warning Notes: {rationale.get('notes_on_seed_warnings')}\n")
            
            f.write("\n")

        f.write("=" * 80 + "\n")
        f.write(f"Total: {len(variants)} expression variants\n")
        f.write("=" * 80 + "\n")


def save_variants_to_json(variants: List[Dict], filepath: Path, dataset_names: List[str], 
                          total_seed_combinations: int):
    """Save variants to JSON file (supports GPT-generated variants)"""
    with open(filepath, 'w', encoding='utf-8') as f:
        json.dump({
            'timestamp': datetime.now().isoformat(),
            'generation_method': 'gpt_based',
            'datasets': dataset_names,
            'total_seed_combinations': total_seed_combinations,
            'total_variants': len(variants),
            'variants': variants
        }, f, indent=2, ensure_ascii=False)

print("✓ Output functions defined")

✓ Output functions defined


## 9. Brain Simulation Functions

In [280]:
def build_alpha_simulation_config(expression: str) -> Dict:
    """Build simulation configuration for Brain API"""
    return {
        "type": "REGULAR",
        "settings": {
            "instrumentType": "EQUITY",
            "region": REGION,
            "universe": UNIVERSE,
            "delay": DELAY,
            "decay": 0,
            "neutralization": "INDUSTRY",
            "truncation": 0.08,
            "pasteurization": "ON",
            "unitHandling": "VERIFY",
            "nanHandling": "OFF",
            "language": "FASTEXPR",
            "testPeriod": "P0Y0M0D",
            "visualization": False,
        },
        "regular": expression,
    }

print("✓ Brain simulation functions defined")

✓ Brain simulation functions defined


## 10. Dataset Files Check

이 셀은 Cell 4에서 자동 감지된 데이터셋 파일들을 확인합니다.

**파일 크기와 존재 여부를 표시**하여 문제를 사전에 발견할 수 있습니다.

In [281]:
print("="*70)
print("DATASET & RESOURCE FILES CHECK")
print("="*70)

# Dataset files (이미 Cell 4에서 로드됨)
print(f"\n📁 Dataset files ({len(DATASET_FILES)} total):")
for name, path in sorted(DATASET_FILES.items()):
    exists = path.exists()
    status = "✓ EXISTS" if exists else "✗ MISSING"
    
    # 파일 크기 표시 (존재하는 경우)
    size_info = ""
    if exists:
        size_bytes = path.stat().st_size
        if size_bytes < 1024:
            size_info = f"{size_bytes} B"
        elif size_bytes < 1024*1024:
            size_info = f"{size_bytes/1024:.1f} KB"
        else:
            size_info = f"{size_bytes/(1024*1024):.1f} MB"
    
    print(f"  {name:15s} [{status:10s}] {size_info:>10s}  {path}")

# Resource files
print(f"\n📚 Resource files:")
resources = [
    ("operators_list.json", OPERATORS_FILE),
    ("datafields.json", DATAFIELDS_FILE)
]

for name, path in resources:
    exists = path.exists()
    status = "✓ EXISTS" if exists else "✗ MISSING"
    
    size_info = ""
    if exists:
        size_bytes = path.stat().st_size
        if size_bytes < 1024:
            size_info = f"{size_bytes} B"
        elif size_bytes < 1024*1024:
            size_info = f"{size_bytes/1024:.1f} KB"
        else:
            size_info = f"{size_bytes/(1024*1024):.1f} MB"
    
    print(f"  {name:25s} [{status:10s}] {size_info:>10s}")
    print(f"    → {path}")

# 선택된 조합 데이터셋 확인
print(f"\n🎯 Datasets selected for combination ({len(DATASETS_TO_COMBINE)}):")
for ds in DATASETS_TO_COMBINE:
    if ds in DATASET_FILES:
        print(f"  ✓ {ds}")
    else:
        print(f"  ✗ {ds} (NOT FOUND)")

print("="*70)

DATASET & RESOURCE FILES CHECK

📁 Dataset files (2 total):
  model25         [✓ EXISTS  ]    27.5 KB  C:\Users\adg01\llm_alpha_gen\llm_alpha_gen\model25.txt
  model30         [✓ EXISTS  ]    14.2 KB  C:\Users\adg01\llm_alpha_gen\llm_alpha_gen\model30.txt

📚 Resource files:
  operators_list.json       [✓ EXISTS  ]    34.5 KB
    → C:\Users\adg01\llm_alpha_gen\llm_alpha_gen\operators_list.json
  datafields.json           [✓ EXISTS  ]     2.0 MB
    → C:\Users\adg01\llm_alpha_gen\llm_alpha_gen\datafield\1\EUR\TOP2500\1_EUR_TOP2500_total.json

🎯 Datasets selected for combination (2):
  ✓ model25
  ✓ model30


## 11. Dry Run: 조합 개수 계산

실제 파이프라인 실행 전에 몇 개의 조합이 생성될지 확인합니다.

In [282]:
print("[Dry Run] Loading zero-fail seeds...\n")

all_alphas = load_zero_fail_alphas(DATASET_FILES)

print("\n" + "="*60)
print("COMBINATION CALCULATION")
print("="*60)

seed_counts = []
for dataset_name in DATASETS_TO_COMBINE:
    if dataset_name in all_alphas:
        count = len(all_alphas[dataset_name])
        seed_counts.append(count)
        print(f"  {dataset_name}: {count} seeds")
    else:
        print(f"  {dataset_name}: NOT FOUND")

if seed_counts:
    total_combinations = 1
    for count in seed_counts:
        total_combinations *= count

    calculation_str = ' × '.join(map(str, seed_counts))
    print(f"\nCartesian Product: {calculation_str} = {total_combinations:,} combinations")

    if MAX_COMBINATIONS is not None:
        actual_combinations = min(total_combinations, MAX_COMBINATIONS)
        print(f"Sampling limit: {MAX_COMBINATIONS:,} → Will generate {actual_combinations:,} combinations")
    else:
        actual_combinations = total_combinations
        print(f"No sampling limit → Will generate ALL {total_combinations:,} combinations")

    total_variants = actual_combinations * MAX_VARIANTS_PER_COMBO
    print(f"\nExpected variants: {actual_combinations:,} combinations × {MAX_VARIANTS_PER_COMBO} variants = {total_variants:,} total")
    
    if total_variants > 100000:
        print(f"\n⚠️  WARNING: {total_variants:,} variants is VERY LARGE!")
        print(f"   Consider setting MAX_COMBINATIONS or reducing MAX_VARIANTS_PER_COMBO")
    elif total_variants > 10000:
        print(f"\n⚠️  WARNING: {total_variants:,} variants may take significant time")
else:
    print("\n[ERROR] No valid datasets found!")

print("="*60)

[Dry Run] Loading zero-fail seeds...

[INFO] model25: 52 zero-fail alphas loaded
[INFO] model30: 29 zero-fail alphas loaded

COMBINATION CALCULATION
  model25: 52 seeds
  model30: 29 seeds

Cartesian Product: 52 × 29 = 1,508 combinations
No sampling limit → Will generate ALL 1,508 combinations

Expected variants: 1,508 combinations × 5 variants = 7,540 total


## 12. Brain Session Login

**시뮬레이션을 실행하려면 이 셀을 먼저 실행하세요!**

Biometrics 인증이 필요할 수 있습니다.

In [293]:
if RUN_SIMULATION:
    print("[INFO] Starting Brain API session...")
    session = ace.start_session()
    print("[INFO] ✓ Session established")
else:
    print("[INFO] Simulation disabled (RUN_SIMULATION = False)")
    print("       Set RUN_SIMULATION = True in Cell 2 if you want to run simulation")
    session = None

[INFO] Starting Brain API session...
Complete biometrics authentication and press any key to continue: 
https://api.worldquantbrain.com/authentication/persona?inquiry=inq_jCTmdSt9AyPKwb3qHR3RvC4rPgm3

[INFO] ✓ Session established


## 13. Step 1: Generate Seed Combinations (Cartesian Product)

In [284]:
print("=" * 80)
print("STEP 1: GENERATING SEED COMBINATIONS")
print("=" * 80)

seed_combinations = generate_all_combinations(
    all_alphas,
    DATASETS_TO_COMBINE,
    max_combinations=MAX_COMBINATIONS,
    random_seed=RANDOM_SEED
)

if seed_combinations:
    print(f"\n✓ Generated {len(seed_combinations):,} seed combinations")
    print(f"\nFirst combination example:")
    print(f"  Datasets: {seed_combinations[0]['datasets']}")
    print(f"  Expressions:")
    for i, expr in enumerate(seed_combinations[0]['expressions']):
        print(f"    [{seed_combinations[0]['datasets'][i]}] {expr[:80]}{'...' if len(expr) > 80 else ''}")
else:
    print("\n✗ Failed to generate combinations")

STEP 1: GENERATING SEED COMBINATIONS
[INFO] Generating Cartesian product from 2 datasets:
  - model25: 52 seeds
  - model30: 29 seeds
[INFO] Total combinations: 1,508


[INFO] Generated 1,508 seed combinations

✓ Generated 1,508 seed combinations

First combination example:
  Datasets: ['model25', 'model30']
  Expressions:
    [model25] ts_zscore(rank(mdl25_vrv421_81v), 84)
    [model30] ts_zscore(star_eps_surprise_prediction_fy2, 5)


## 14. Step 2: Generate Variants (GPT-Based)

**NEW!** GPT를 사용하여 각 seed 조합에 대해 여러 variant를 생성합니다.

### PowerPool Alpha 회피 전략

GPT에게 다음을 명시적으로 지시:
1. **Cross-sectional normalization** (rank, zscore, quantile) - weight를 고르게 분산
2. **Group-based normalization** (group_zscore with bucket grouping) - peer group 내 정규화
3. **Tail handling** (winsorize, pasteurize) - extreme weight 제한
4. **Weighted combinations** - 여러 signal 균형있게 조합

### 품질 보장

- **operator_list.json 강제**: GPT가 허용된 operator만 사용
- **JSON 응답 파싱**: robust parser로 안정적 파싱
- **Deduplication**: 중복 expression 제거
- **Sanity Check**: Step 3에서 type validation

### 실행 흐름

1. 각 seed combination에 대해 GPT 호출
2. PowerPool 회피 전략이 포함된 프롬프트 전송
3. JSON 응답 파싱 및 variant 생성
4. 중복 제거 및 통계 출력

In [285]:
print("=" * 80)
print("STEP 2: GENERATING VARIANTS (GPT-BASED)")
print("=" * 80)

# Load operators
operators = llm.import_json(str(OPERATORS_FILE))
print(f"[INFO] Loaded {len(operators)} operators from {OPERATORS_FILE.name}")
print(f"[INFO] Using GPT model: {GPT_MODEL}")
print(f"[INFO] Requesting {MAX_VARIANTS_PER_COMBO} variants per combination")

# Generate variants using GPT
all_variants = []
gpt_call_count = 0
gpt_success_count = 0
gpt_failure_count = 0

for i, seed_combo in enumerate(seed_combinations):
    print(f"\n[{i+1}/{len(seed_combinations)}] Processing combination:")
    for j, ds in enumerate(seed_combo['datasets']):
        expr_preview = seed_combo['expressions'][j][:60]
        print(f"  [{ds}] {expr_preview}{'...' if len(seed_combo['expressions'][j]) > 60 else ''}")
    
    # Call GPT to generate variants
    gpt_call_count += 1
    variants = generate_combination_variants(
        seed_combo,
        operators,
        max_variants_per_combo=MAX_VARIANTS_PER_COMBO,
        random_seed=RANDOM_SEED,
        model=GPT_MODEL
    )
    
    if variants:
        gpt_success_count += 1
        all_variants.extend(variants)
        print(f"    ✓ Added {len(variants)} variants to pool")
    else:
        gpt_failure_count += 1
        print(f"    ✗ GPT failed to generate variants")
    
    # Progress logging every 10 combinations
    if (i + 1) % 10 == 0:
        print(f"\n[PROGRESS] Processed {i + 1:,}/{len(seed_combinations):,} combinations")
        print(f"           Total variants so far: {len(all_variants):,}")
        print(f"           GPT success: {gpt_success_count}/{gpt_call_count}, failures: {gpt_failure_count}")

# Deduplication by expression
print(f"\n{'='*60}")
print("DEDUPLICATION")
print(f"{'='*60}")
unique_variants = []
seen_expressions = set()

for variant in all_variants:
    expr = variant['expression'].strip()
    if expr and expr not in seen_expressions:
        seen_expressions.add(expr)
        unique_variants.append(variant)

duplicates_removed = len(all_variants) - len(unique_variants)
print(f"  Original variants: {len(all_variants):,}")
print(f"  Unique variants: {len(unique_variants):,}")
print(f"  Duplicates removed: {duplicates_removed:,}")

all_variants = unique_variants

# Summary statistics
print(f"\n{'='*60}")
print("GENERATION SUMMARY")
print(f"{'='*60}")
print(f"  Total combinations processed: {len(seed_combinations):,}")
print(f"  GPT calls: {gpt_call_count}")
print(f"  GPT success: {gpt_success_count} ({100*gpt_success_count/gpt_call_count if gpt_call_count > 0 else 0:.1f}%)")
print(f"  GPT failures: {gpt_failure_count}")
print(f"  Total unique variants: {len(all_variants):,}")
print(f"  Average variants per combination: {len(all_variants)/len(seed_combinations) if len(seed_combinations) > 0 else 0:.1f}")

# Variant type distribution (idea summary)
if all_variants:
    print(f"\nTop 10 variant ideas:")
    idea_counts = {}
    for v in all_variants:
        idea = v.get('idea', 'N/A')[:50]  # Truncate long ideas
        idea_counts[idea] = idea_counts.get(idea, 0) + 1
    
    for idea, count in sorted(idea_counts.items(), key=lambda x: -x[1])[:10]:
        print(f"  {idea:50s}: {count:,}")

print(f"{'='*60}")

STEP 2: GENERATING VARIANTS (GPT-BASED)
[INFO] Loaded 83 operators from operators_list.json
[INFO] Using GPT model: gpt-4o-mini
[INFO] Requesting 5 variants per combination

[1/1508] Processing combination:
  [model25] ts_zscore(rank(mdl25_vrv421_81v), 84)
  [model30] ts_zscore(star_eps_surprise_prediction_fy2, 5)
    [GPT] Generating 5 combinations...
{
  "combinations": [
    {
      "expression": "rank(add(ts_zscore(rank(mdl25_vrv421_81v), 84), ts_zscore(star_eps_surprise_prediction_fy2, 5)))",
      "operators_used": ["rank", "add", "ts_zscore"],
      "datasets_used": ["model25", "model30"],
      "idea": "Combine z-scores to balance signals from both models",
      "rationale": {
        "why_this_helps_powerpool": "Combining z-scores evens out the influence of individual signals.",
        "how_it_reduces_concentration": "Using rank after combining distributes weights across the universe.",
        "notes_on_seed_warnings": "By aggregating the z-scores, no single stock can domin

KeyboardInterrupt: 

## 15. Step 3: Sanity Check

Type validation으로 invalid expression을 제거합니다.

In [286]:
if CHECK_SANITY:
    print("=" * 80)
    print("STEP 3: SANITY CHECK (Type Validation)")
    print("=" * 80)

    # Check if all_variants exists and is not empty
    if 'all_variants' not in locals() or len(all_variants) == 0:
        print("[ERROR] No variants to check!")
        print("        Make sure you ran Cell 28 (Step 2: Generate Variants) first")
        if 'all_variants' in locals():
            print(f"        Current all_variants length: {len(all_variants)}")
        else:
            print(f"        Variable 'all_variants' is undefined")
        print("\n⚠️  STOPPING: Cannot proceed with sanity check")
    else:
        # Load datafields
        datafields = llm.import_json(str(DATAFIELDS_FILE))
        print(f"[INFO] Loaded {len(datafields)} datafields from {DATAFIELDS_FILE.name}")
        print(f"[INFO] Loaded {len(operators)} operators for validation")

        valid_variants = []
        sanity_failures = {}
        unknown_operators = set()

        for i, variant in enumerate(all_variants):
            is_valid, error_msg = sanity_checker(
                variant['expression'],
                operators,
                datafields
            )

            if is_valid:
                valid_variants.append(variant)
            else:
                reason = error_msg or "unknown"
                sanity_failures[reason] = sanity_failures.get(reason, 0) + 1
                
                # Track unknown operators specifically
                if "Unknown operator:" in reason:
                    op_name = reason.replace("Unknown operator:", "").strip()
                    unknown_operators.add(op_name)

            if (i + 1) % 5000 == 0:
                print(f"  Checked {i + 1:,}/{len(all_variants):,} variants...")

        print(f"\n{'='*60}")
        print("VALIDATION RESULTS")
        print(f"{'='*60}")
        print(f"  Total variants checked: {len(all_variants):,}")
        
        # Safe percentage calculation
        if len(all_variants) > 0:
            print(f"  Valid: {len(valid_variants):,} ({100*len(valid_variants)/len(all_variants):.1f}%)")
            print(f"  Failed: {len(all_variants) - len(valid_variants):,} ({100*(len(all_variants) - len(valid_variants))/len(all_variants):.1f}%)")
        else:
            print(f"  Valid: 0 (N/A)")
            print(f"  Failed: 0")

        if unknown_operators:
            print(f"\n⚠️  UNKNOWN OPERATORS DETECTED (GPT hallucination):")
            print(f"  Total unique unknown operators: {len(unknown_operators)}")
            for op in sorted(unknown_operators):
                count = sanity_failures.get(f"Unknown operator: {op}", 0)
                print(f"    - {op} ({count} occurrences)")
            print(f"\n  → These operators are NOT in operators_list.json!")
            print(f"  → GPT may need stricter prompt constraints or better operator list formatting")

        if sanity_failures:
            print(f"\nFailure breakdown (top 10 reasons):")
            for reason, count in sorted(sanity_failures.items(), key=lambda x: -x[1])[:10]:
                print(f"  {reason[:70]:70s}: {count:,}")

        all_variants = valid_variants
        print(f"\n✓ Sanity check complete - keeping {len(valid_variants):,} valid variants")
else:
    print("[INFO] Sanity check SKIPPED (CHECK_SANITY = False)")

STEP 3: SANITY CHECK (Type Validation)
[INFO] Loaded 1908 datafields from 1_EUR_TOP2500_total.json
[INFO] Loaded 83 operators for validation

VALIDATION RESULTS
  Total variants checked: 115
  Valid: 85 (73.9%)
  Failed: 30 (26.1%)

Failure breakdown (top 10 reasons):
  Parsing error: pop from empty list                                    : 28
  Parsing error: 'group'                                                : 2

✓ Sanity check complete - keeping 85 valid variants


## 16. Step 4: Save Results

In [287]:
print("=" * 80)
print("STEP 4: SAVING RESULTS")
print("=" * 80)

# Check if we have variants to save
if 'all_variants' not in locals() or len(all_variants) == 0:
    print("[ERROR] No variants to save!")
    print("        Make sure you completed previous steps:")
    print("        - Cell 26: Generate seed combinations")
    print("        - Cell 28: Generate variants (GPT)")
    print("        - Cell 30: Sanity check (if enabled)")
    if 'all_variants' in locals():
        print(f"        Current all_variants length: {len(all_variants)}")
    else:
        print(f"        Variable 'all_variants' is undefined")
    print("\n⚠️  STOPPING: Cannot save empty results")
else:
    # Create output directory
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    # Generate filename
    dataset_combo_name = '_'.join(sorted(DATASETS_TO_COMBINE))
    txt_file = OUTPUT_DIR / f"{dataset_combo_name}_comb.txt"
    json_file = OUTPUT_DIR / f"{dataset_combo_name}_comb.json"

    # Save text format
    print(f"[INFO] Saving to {txt_file}...")
    save_variants_to_txt(all_variants, txt_file, DATASETS_TO_COMBINE)

    # Save JSON format
    print(f"[INFO] Saving to {json_file}...")
    save_variants_to_json(
        all_variants, 
        json_file, 
        DATASETS_TO_COMBINE,
        len(seed_combinations) if 'seed_combinations' in locals() else 0
    )

    print(f"\n✓ Results saved:")
    print(f"  TXT:  {txt_file}")
    print(f"  JSON: {json_file}")

STEP 4: SAVING RESULTS
[INFO] Saving to C:\Users\adg01\llm_alpha_gen\llm_alpha_gen\results\combinatorial\model25_model30_comb.txt...
[INFO] Saving to C:\Users\adg01\llm_alpha_gen\llm_alpha_gen\results\combinatorial\model25_model30_comb.json...

✓ Results saved:
  TXT:  C:\Users\adg01\llm_alpha_gen\llm_alpha_gen\results\combinatorial\model25_model30_comb.txt
  JSON: C:\Users\adg01\llm_alpha_gen\llm_alpha_gen\results\combinatorial\model25_model30_comb.json


## 17. Summary

In [288]:
print("=" * 80)
print("PIPELINE SUMMARY")
print("=" * 80)
print(f"Datasets combined: {', '.join(DATASETS_TO_COMBINE)}")

if 'seed_combinations' in locals():
    print(f"Seed combinations: {len(seed_combinations):,}")
else:
    print(f"Seed combinations: NOT GENERATED")

if 'all_variants' in locals():
    print(f"Variants generated: {len(all_variants):,}")
else:
    print(f"Variants generated: NOT GENERATED")

print(f"Sanity check: {'PASS' if CHECK_SANITY else 'SKIPPED'}")
print(f"GPT Model: {GPT_MODEL}")
print(f"")

# Check if output files exist
dataset_combo_name = '_'.join(sorted(DATASETS_TO_COMBINE))
txt_file = OUTPUT_DIR / f"{dataset_combo_name}_comb.txt"
json_file = OUTPUT_DIR / f"{dataset_combo_name}_comb.json"

if txt_file.exists() and json_file.exists():
    print(f"Output files:")
    print(f"  {txt_file}")
    print(f"  {json_file}")
else:
    print(f"Output files: NOT CREATED YET")
    print(f"  Expected locations:")
    print(f"  {txt_file}")
    print(f"  {json_file}")

print("=" * 80)

PIPELINE SUMMARY
Datasets combined: model25, model30
Seed combinations: 1,508
Variants generated: 85
Sanity check: PASS
GPT Model: gpt-4o-mini

Output files:
  C:\Users\adg01\llm_alpha_gen\llm_alpha_gen\results\combinatorial\model25_model30_comb.txt
  C:\Users\adg01\llm_alpha_gen\llm_alpha_gen\results\combinatorial\model25_model30_comb.json


## 18. (Optional) Brain API Simulation

**주의**: 많은 variant를 시뮬레이션하면 시간과 리소스가 많이 소모됩니다!

시뮬레이션 전에:
1. Cell 2에서 `RUN_SIMULATION = True` 설정
2. Cell 12에서 Brain 세션 로그인
3. 이 셀 실행

In [ ]:
if RUN_SIMULATION and session is not None:
    print("=" * 80)
    print("BRAIN API SIMULATION")
    print("=" * 80)
    print(f"[WARNING] Simulating {len(all_variants):,} alphas may take a long time!\n")
    
    # Build alpha configs
    alpha_list = []
    for variant in all_variants:
        alpha_config = build_alpha_simulation_config(variant['expression'])
        alpha_list.append(alpha_config)
    
    # Simulate
    print(f"[INFO] Starting simulation with concurrency={CONCURRENCY}...")
    results = ace.simulate_alpha_list_multi(
        session,
        alpha_list,
        limit_of_concurrent_simulations=CONCURRENCY,
        limit_of_multi_simulations=CONCURRENCY,
        simulation_config={
            "check_submission": True,
            "get_pnl": False,
            "get_stats": False,
        }
    )
    
    print(f"\n✓ Simulation complete: {len(results)} results")
    
    # Save simulation results
    sim_result_file = OUTPUT_DIR / f"{dataset_combo_name}_simulation_results.json"
    
    print(f"[INFO] Saving simulation results to {sim_result_file}...")
    
    with open(sim_result_file, 'w', encoding='utf-8') as f:
        json.dump({
            'timestamp': datetime.now().isoformat(),
            'datasets': DATASETS_TO_COMBINE,
            'total_simulated': len(results),
            'results': [{
                'alpha_id': r.get('alpha_id'),
                'expression': r.get('simulate_data', {}).get('regular'),
                'is_sharpe': float(r.get('is_stats', pd.DataFrame()).iloc[0].get('sharpe', 0) if not r.get('is_stats', pd.DataFrame()).empty else 0),
                'is_fitness': float(r.get('is_stats', pd.DataFrame()).iloc[0].get('fitness', 0) if not r.get('is_stats', pd.DataFrame()).empty else 0),
                'is_turnover': float(r.get('is_stats', pd.DataFrame()).iloc[0].get('turnover', 0) if not r.get('is_stats', pd.DataFrame()).empty else 0),
            } for r in results]
        }, f, indent=2, ensure_ascii=False)
    
    print(f"✓ Simulation results saved to {sim_result_file}")
    
    # Print top performers
    print(f"\nTop 10 performers by Sharpe:")
    sorted_results = sorted(
        [r for r in results if r.get('is_stats') is not None and not r.get('is_stats').empty],
        key=lambda x: float(x['is_stats'].iloc[0].get('sharpe', 0) or 0),
        reverse=True
    )[:10]
    
    for i, r in enumerate(sorted_results, 1):
        stats = r['is_stats'].iloc[0]
        expr = r.get('simulate_data', {}).get('regular', '')
        print(f"  #{i} Sharpe: {stats.get('sharpe', 0):.2f}, Fitness: {stats.get('fitness', 0):.2f}")
        print(f"      {expr[:100]}{'...' if len(expr) > 100 else ''}")

elif RUN_SIMULATION and session is None:
    print("[ERROR] Simulation enabled but session not established!")
    print("        Run Cell 12 first to login to Brain API")
else:
    print("[INFO] Simulation skipped (RUN_SIMULATION = False)")
    print("       Set RUN_SIMULATION = True in Cell 2 if you want to run simulation")

BRAIN API SIMULATION
[WARNING] Simulating 85 alphas may take a long time!

[INFO] Starting simulation with concurrency=3...


  0%|          | 0/29 [00:00<?, ?it/s]2026-02-08 17:16:57,406 - ace - WARNING - Child-Simulation 28kUhIgdi4o0bTVB3ElqHHc failed: Unknown error. Full response: {'id': '28kUhIgdi4o0bTVB3ElqHHc', 'type': 'REGULAR', 'status': 'CANCELLED'}


  [CHILD FAIL] ID=28kUhIgdi4o0bTVB3ElqHHc: Unknown error


2026-02-08 17:16:57,693 - ace - WARNING - Child-Simulation 2xw6pL1zM4ww9lROXNuXhWa failed: Required attribute "lookback" must have a value. Full response: {'id': '2xw6pL1zM4ww9lROXNuXhWa', 'type': 'REGULAR', 'status': 'ERROR', 'message': 'Required attribute "lookback" must have a value', 'location': {'line': 1, 'start': 0, 'end': 106, 'property': 'regular'}}


  [CHILD FAIL] ID=2xw6pL1zM4ww9lROXNuXhWa: Required attribute "lookback" must have a value


...

  [CHILD FAIL] ID=14Mhkec1E4y99VxpeDry19H: Unknown error


2026-02-08 17:23:27,491 - ace - INFO - Multi-simulation completed: 3/3 successful
100%|██████████| 29/29 [07:39<00:00, 15.85s/it]


Complete biometrics authentication and press any key to continue: 
https://api.worldquantbrain.com/authentication/persona?inquiry=inq_UAX4yuR7y4EAzwK4ibdgEssqxuhL



2026-02-08 17:28:48,161 - ace - INFO - Trying re-login, wait 100 seconds


Complete biometrics authentication and press any key to continue: 
https://api.worldquantbrain.com/authentication/persona?inquiry=inq_FhhdyEW43iLvnpL7HLRX4zpu9rh7



## Done!

---

## 📚 데이터셋 관리 가이드

### 🚀 Quick Start (AUTO 모드)

**Step 1**: 데이터셋 파일을 프로젝트 폴더에 배치
```
llm_alpha_gen/
  ├── model25.txt
  ├── model30.txt
  ├── model138.txt
  └── combine_and_simulate.ipynb
```

**Step 2**: Cell 4에서 AUTO 모드 활성화
```python
AUTO_DISCOVER_DATASETS = True
DATASET_PATTERNS = ['model*.txt', 'nws*.txt']
DATASETS_TO_COMBINE = None  # 모든 발견된 데이터셋 사용
```

**Step 3**: Cell 4 실행 → 자동으로 3개 파일 발견!

---

### 🎯 사용 시나리오

#### 시나리오 1: 모든 model 데이터셋 조합
```python
AUTO_DISCOVER_DATASETS = True
DATASET_PATTERNS = ['model*.txt']  # model로 시작하는 파일만
DATASETS_TO_COMBINE = None  # 모두 사용
```

#### 시나리오 2: 특정 3개 데이터셋만 조합
```python
AUTO_DISCOVER_DATASETS = True
DATASET_PATTERNS = ['model*.txt', 'nws*.txt']  # 모두 찾되
DATASETS_TO_COMBINE = ['model25', 'nws17', 'model138']  # 이 3개만 조합
```

#### 시나리오 3: 매번 다른 조합 실험
```python
# 실험 1
DATASETS_TO_COMBINE = ['model25', 'model30']

# 실험 2 (파일만 수정)
DATASETS_TO_COMBINE = ['model25', 'nws17']

# 실험 3
DATASETS_TO_COMBINE = ['model138', 'nws17', 'model25']
```

#### 시나리오 4: 하위 폴더에서 찾기
```python
DATASET_SEARCH_DIR = SCRIPT_DIR / "datasets"  # 하위 폴더 지정
DATASET_PATTERNS = ['*.txt']  # 모든 .txt 파일
```

---

### ⚙️ 설정 옵션 상세

| 옵션 | 설명 | 예시 |
|------|------|------|
| `AUTO_DISCOVER_DATASETS` | 자동 감지 ON/OFF | `True` / `False` |
| `DATASET_SEARCH_DIR` | 검색 디렉토리 | `SCRIPT_DIR` or `SCRIPT_DIR / "data"` |
| `DATASET_PATTERNS` | 파일 패턴 (리스트) | `['model*.txt', 'nws*.txt']` |
| `DATASETS_TO_COMBINE` | 조합할 데이터셋 | `None` (모두) or `['model25', 'model30']` |
| `MANUAL_DATASET_FILES` | 수동 모드용 딕셔너리 | `{'model25': Path('model25.txt')}` |

---

### 🔍 자주 묻는 질문

**Q: 새 데이터셋 추가하려면?**  
A: 파일만 프로젝트 폴더에 복사하면 끝! Cell 4 재실행 시 자동 감지됩니다.

**Q: 특정 데이터셋 제외하려면?**  
A: `DATASETS_TO_COMBINE`에서 제거하거나, `DATASET_PATTERNS`에서 패턴 조정.

**Q: 파일 이름이 패턴에 안 맞으면?**  
A: `DATASET_PATTERNS`에 새 패턴 추가. 예: `['*.txt']` (모든 txt 파일)

**Q: 수동 모드로 돌아가려면?**  
A: `AUTO_DISCOVER_DATASETS = False` 설정

---

### 📝 실제 파일명 예시

현재 프로젝트에서 사용하는 파일명 형태:
- `model25.txt` ✓
- `model30.txt` ✓
- `model138.txt` ✓
- `nws17.txt` ✓

**주의**: 
- ~~`mdl25.txt`~~ (구버전, 사용 안 함)
- `model25.txt` (현재 사용)

---

### Next Steps

1. **결과 검토**: `results/combinatorial/` 폴더의 파일들 확인
2. **Simulation**: `RUN_SIMULATION = True` 설정 후 Cell 12, 18 실행
3. **파라미터 조정**: Cell 2의 설정값 변경 후 재실행
4. **새 Dataset 추가**: 파일만 추가하면 자동 감지!